# TheQA Tutorial: Critical Noise Thresholds in Discrete Dynamical Systems

This notebook provides an interactive introduction to TheQA package.

## 1. Installation and Setup

First, make sure TheQA is installed:

In [ ]:
# Install TheQA (if not already installed)
# !pip install theqa

# Import required packages
import numpy as np
import matplotlib.pyplot as plt
from theqa import compute_sigma_c, TripleRule
from theqa.systems import CollatzSystem, FibonacciSystem, LogisticMap
from theqa.features import PeakCounter, EntropyCalculator
from theqa.criteria import VarianceCriterion

# Set up plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
print("TheQA loaded successfully!")

## 2. Basic Usage: Computing σc

Let's start with a simple example - computing the critical noise threshold for a sequence.

In [ ]:
# Example 1: Simple exponential sequence
sequence = [1, 2, 4, 8, 16, 32, 64, 128, 256]
sigma_c = compute_sigma_c(sequence)
print(f"Critical threshold for exponential sequence: σc = {sigma_c:.3f}")

# Example 2: Collatz sequence
collatz = CollatzSystem(n=27)
collatz_seq = collatz.generate()
sigma_c_collatz = compute_sigma_c(collatz_seq)
print(f"Critical threshold for Collatz(27): σc = {sigma_c_collatz:.3f}")

# Visualize the Collatz sequence
plt.figure(figsize=(10, 4))
plt.plot(collatz_seq, 'b-', linewidth=1)
plt.xlabel('Step')
plt.ylabel('Value')
plt.title(f'Collatz Sequence starting from 27 (σc = {sigma_c_collatz:.3f})')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## 3. The Triple Rule: σc(S, F, C)

The critical threshold depends on three components:
- **System (S)**: The mathematical sequence
- **Feature (F)**: What we measure
- **Criterion (C)**: How we detect changes

In [ ]:
# Create a system
system = CollatzSystem(n=27)

# Try different features
features = {
    'Peak Count': PeakCounter(transform='log'),
    'Entropy': EntropyCalculator(bins=20),
    'Peak Count (no transform)': PeakCounter(transform=None)
}

# Compute σc for each feature
print("Effect of different features on σc:")
print("-" * 40)

for name, feature in features.items():
    tr = TripleRule(
        system=system,
        feature=feature,
        criterion=VarianceCriterion(threshold=0.1)
    )
    result = tr.compute(n_trials=50, method='adaptive')
    print(f"{name}: σc = {result.sigma_c:.3f}")

## 4. Visualizing Phase Transitions

Let's visualize how the system behavior changes as we increase the noise level.

In [ ]:
# Generate sequence
sequence = CollatzSystem(n=27).generate()
log_seq = np.log(sequence + 1)

# Test different noise levels
sigmas = np.logspace(-3, 0, 30)
variances = []

for sigma in sigmas:
    peak_counts = []
    for _ in range(100):
        noise = np.random.normal(0, sigma, len(log_seq))
        noisy = log_seq + noise
        peaks = len(signal.find_peaks(noisy, prominence=sigma/2)[0])
        peak_counts.append(peaks)
    variances.append(np.var(peak_counts))

# Plot phase transition
plt.figure(figsize=(10, 6))
plt.semilogx(sigmas, variances, 'b-', linewidth=2)
plt.axvline(x=0.117, color='r', linestyle='--', linewidth=2, label='σc = 0.117')
plt.xlabel('Noise Level (σ)')
plt.ylabel('Feature Variance')
plt.title('Phase Transition in Collatz System')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Show examples at different noise levels
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
noise_levels = [0.01, 0.1, 0.117, 0.5]

for idx, (ax, sigma) in enumerate(zip(axes.flat, noise_levels)):
    noise = np.random.normal(0, sigma, len(log_seq))
    noisy = log_seq + noise
    
    ax.plot(noisy, 'b-', alpha=0.7, linewidth=1)
    ax.set_title(f'σ = {sigma:.3f}')
    ax.set_xlabel('Step')
    ax.set_ylabel('Log Value')
    ax.grid(True, alpha=0.3)

plt.suptitle('Collatz Sequence with Different Noise Levels')
plt.tight_layout()
plt.show()

## 5. Comparing Different Systems

Different mathematical systems have vastly different critical thresholds.

In [ ]:
# Compare different systems
systems = {
    'Fibonacci': FibonacciSystem(n=50),
    'Collatz': CollatzSystem(n=27),
    'Logistic (r=3.6)': LogisticMap(r=3.6, length=200),
    'Logistic (r=4.0)': LogisticMap(r=4.0, length=200)
}

results = {}

for name, system in systems.items():
    sigma_c = compute_sigma_c(system.generate(), method='adaptive')
    results[name] = sigma_c
    print(f"{name}: σc = {sigma_c:.3f}")

# Visualize comparison
plt.figure(figsize=(10, 6))
names = list(results.keys())
values = list(results.values())
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

bars = plt.bar(names, values, color=colors, alpha=0.7)
plt.axhline(y=np.pi/2, color='red', linestyle='--', linewidth=2, label='π/2 bound')

# Add value labels on bars
for bar, value in zip(bars, values):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{value:.3f}', ha='center', va='bottom')

plt.ylabel('Critical Threshold (σc)')
plt.title('Critical Thresholds for Different Systems')
plt.ylim(0, 1.8)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.show()

# Classification
print("\nUniversality Classes:")
for name, sigma_c in results.items():
    if sigma_c < 0.01:
        class_name = "Ultra-sensitive"
    elif sigma_c < 0.1:
        class_name = "Sensitive"
    elif sigma_c < 0.3:
        class_name = "Medium"
    else:
        class_name = "Robust"
    print(f"{name}: {class_name} (σc = {sigma_c:.3f})")

## 6. Efficient Algorithms

TheQA provides several algorithms with different speed/accuracy trade-offs.

In [ ]:
from theqa.algorithms import spectral_sigma_c, gradient_sigma_c, analytical_sigma_c
import time

# Generate a test sequence
sequence = CollatzSystem(n=27).generate(max_steps=1000)

# Compare methods
methods = {
    'Empirical': lambda seq: compute_sigma_c(seq, method='empirical', n_trials=50),
    'Spectral': spectral_sigma_c,
    'Gradient': gradient_sigma_c,
    'Analytical': analytical_sigma_c
}

print("Algorithm Comparison:")
print("-" * 50)
print(f"{'Method':<15} {'σc':<10} {'Time (ms)':<15} {'Speedup':<10}")
print("-" * 50)

times = {}
values = {}

for name, method in methods.items():
    start = time.time()
    sigma_c = method(sequence)
    elapsed = (time.time() - start) * 1000  # Convert to ms
    
    times[name] = elapsed
    values[name] = sigma_c
    
    speedup = times.get('Empirical', elapsed) / elapsed if name != 'Empirical' else 1.0
    print(f"{name:<15} {sigma_c:<10.3f} {elapsed:<15.1f} {speedup:<10.1f}x")

## 7. Optimization: Finding Best (F,C) Pairs

For specific applications, we can optimize the choice of feature and criterion.

In [ ]:
# Test different combinations for Collatz system
system = CollatzSystem(n=27)

from theqa.features import SpectralAnalyzer, AutoCorrelation
from theqa.criteria import IQRCriterion, EntropyCriterion

# Define features and criteria to test
features = {
    'Peaks': PeakCounter(transform='log'),
    'Entropy': EntropyCalculator(bins=20),
    'Spectral': SpectralAnalyzer()
}

criteria = {
    'Variance': VarianceCriterion(threshold=0.1),
    'IQR': IQRCriterion(threshold=0.2),
    'Entropy': EntropyCriterion(threshold=0.1)
}

# Test all combinations
results = []

for f_name, feature in features.items():
    for c_name, criterion in criteria.items():
        tr = TripleRule(system=system, feature=feature, criterion=criterion)
        result = tr.compute(n_trials=30, method='adaptive')
        results.append({
            'Feature': f_name,
            'Criterion': c_name,
            'σc': result.sigma_c
        })

# Convert to DataFrame for nice display
import pandas as pd
df = pd.DataFrame(results)
pivot = df.pivot(index='Feature', columns='Criterion', values='σc')

# Heatmap visualization
plt.figure(figsize=(8, 6))
im = plt.imshow(pivot.values, cmap='viridis', aspect='auto')
plt.colorbar(im, label='σc')

# Add labels
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel('Criterion')
plt.ylabel('Feature')
plt.title('σc Values for Different (F,C) Combinations')

# Add text annotations
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        plt.text(j, i, f'{pivot.values[i, j]:.3f}',
                ha='center', va='center', color='white')

plt.tight_layout()
plt.show()

# Find extremes
min_idx = np.argmin(df['σc'])
max_idx = np.argmax(df['σc'])

print(f"\nMost sensitive (min σc): {df.iloc[min_idx]['Feature']} + {df.iloc[min_idx]['Criterion']} = {df.iloc[min_idx]['σc']:.3f}")
print(f"Most robust (max σc): {df.iloc[max_idx]['Feature']} + {df.iloc[max_idx]['Criterion']} = {df.iloc[max_idx]['σc']:.3f}")

## 8. Quantum Extensions (Optional)

If quantum features are installed, we can explore quantum systems with extended bounds.

In [ ]:
try:
    from theqa.quantum import QuantumWalk, classical_to_quantum_bound
    
    # Classical vs Quantum bounds
    classical_values = [0.117, 0.21, 0.45]
    names = ['Collatz', 'Logistic', 'Random Walk']
    
    print("Classical vs Quantum Bounds:")
    print("-" * 40)
    
    for name, classical in zip(names, classical_values):
        quantum = classical_to_quantum_bound(classical)
        enhancement = quantum / classical
        print(f"{name}: Classical={classical:.3f}, Quantum={quantum:.3f}, Enhancement={enhancement:.2f}x")
    
    # Visualize
    plt.figure(figsize=(10, 6))
    x = np.arange(len(names))
    width = 0.35
    
    plt.bar(x - width/2, classical_values, width, label='Classical', color='blue', alpha=0.7)
    quantum_values = [classical_to_quantum_bound(c) for c in classical_values]
    plt.bar(x + width/2, quantum_values, width, label='Quantum', color='red', alpha=0.7)
    
    plt.axhline(y=np.pi/2, color='blue', linestyle='--', alpha=0.5, label='π/2 (classical bound)')
    plt.axhline(y=np.pi, color='red', linestyle='--', alpha=0.5, label='π (quantum bound)')
    
    plt.xlabel('System')
    plt.ylabel('Critical Threshold')
    plt.title('Classical vs Quantum Critical Thresholds')
    plt.xticks(x, names)
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    plt.show()
    
except ImportError:
    print("Quantum module not available. Install with: pip install theqa[quantum]")

## 9. Summary and Next Steps

You've learned how to:
1. Compute critical noise thresholds (σc)
2. Use the Triple Rule framework
3. Visualize phase transitions
4. Compare different systems
5. Use efficient algorithms
6. Optimize (F,C) pairs for specific goals

### Next steps:
- Explore more systems in `theqa.systems`
- Try custom features and criteria
- Analyze your own sequences
- Read the paper for theoretical background

### Useful resources:
- Documentation: https://theqa.readthedocs.io
- GitHub: https://github.com/hermannhart/theqa
- Paper: arXiv link (when available)

In [ ]:
# Final example: Analyze your own sequence
# Try modifying this!

# Create a custom sequence (example: powers of 3)
custom_sequence = [3**i for i in range(20)]

# Analyze it
sigma_c = compute_sigma_c(custom_sequence)
print(f"Your custom sequence has σc = {sigma_c:.3f}")

# Classify it
if sigma_c < 0.01:
    print("Classification: Ultra-sensitive - highly structured sequence")
elif sigma_c < 0.1:
    print("Classification: Sensitive - moderate structure")
elif sigma_c < 0.3:
    print("Classification: Medium - balanced structure and randomness")
else:
    print("Classification: Robust - resistant to noise")